# Lezione 8 · Attention e Transformer, la tecnica regina

Il giro al mercato: confrontare, dividere le cento monete, raccogliere. Prima per una parola sola, poi per tutte insieme, in forma di matrici; poi la maschera, la bussola che gira e i pesi del blocco.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def softmax(z):
    e = np.exp(np.array(z, dtype=float) - np.max(z))
    return e / e.sum()

## Il mercato in una riga

*Nel racconto: la lista, il cartello, la merce; «confronti, dividi, raccogli».*

Prima il giro di «pesca» nella frase «Ieri, sulla barca, la pesca...», come nel racconto; poi la ragione del √d_k.

In [2]:
lista = np.array([1, 1])                                   # «pesca» chiede a tutte e due: mare e frutta
cartelli = {"barca": np.array([2, 0]), "ieri": np.array([0, 0])}
merci = {"barca": np.array([1, 0]), "ieri": np.array([0, 0])}
punteggi = np.array([lista @ cartelli["barca"], lista @ cartelli["ieri"]])
conta("punteggio con «barca»", punteggi[0], 2)
conta("punteggio con «ieri»", punteggi[1], 0)
monete = softmax(punteggi) * 100
controlla("monete a «barca»", monete[0], 88, 0.5)
controlla("monete a «ieri»", monete[1], 12, 0.5)
cesto = monete[0] / 100 * merci["barca"] + monete[1] / 100 * merci["ieri"]
controlla("il cesto di «pesca», verso il mare", cesto[0], 0.88)

# dividendo per √2, come nella formula
controlla("2 / √2", 2 / math.sqrt(2), 1.41)
monete = softmax(punteggi / math.sqrt(2)) * 100
controlla("monete a «barca», con la divisione", monete[0], 80, 0.5)
controlla("monete a «ieri», con la divisione", monete[1], 20, 0.5)

# perché √d_k: con d_k = 64 i punteggi tipici stanno intorno a ±8
rng = np.random.default_rng(0)
prodotti = np.sum(rng.standard_normal((20000, 64)) * rng.standard_normal((20000, 64)), axis=1)
controlla("d_k = 64: i prodotti scalari oscillano di circa", prodotti.std(), 8, 0.2)
controlla("softmax su (8; 0): tutte le monete su un banco", softmax([8, 0])[0], 0.9997, 0.0001)
controlla("dividendo per 8, softmax su (1; 0)", softmax([1, 0])[0], 0.73)

✓  punteggio con «barca»: calcolato 2, nel libro 2
✓  punteggio con «ieri»: calcolato 0, nel libro 0
✓  monete a «barca»: calcolato 88.08, nel libro 88
✓  monete a «ieri»: calcolato 11.92, nel libro 12
✓  il cesto di «pesca», verso il mare: calcolato 0.8808, nel libro 0.88
✓  2 / √2: calcolato 1.414, nel libro 1.41
✓  monete a «barca», con la divisione: calcolato 80.44, nel libro 80
✓  monete a «ieri», con la divisione: calcolato 19.56, nel libro 20
✓  d_k = 64: i prodotti scalari oscillano di circa: calcolato 8.037, nel libro 8
✓  softmax su (8; 0): tutte le monete su un banco: calcolato 0.9997, nel libro 0.9997
✓  dividendo per 8, softmax su (1; 0): calcolato 0.7311, nel libro 0.73


## Il mercato tutto insieme, in forma di matrici

*Nel racconto: «Ieri, sulla barca, la pesca».*

Tre parole, ognuna con la sua lista, il suo cartello e la sua merce; ogni parola guarda tutte le altre, anche se stessa, senza maschera.

In [3]:
parole = ["ieri", "barca", "pesca"]
Q = np.array([[0, 0], [1, 0], [1, 1]], dtype=float)        # le liste
K = np.array([[0, 0], [2, 0], [1, 1]], dtype=float)        # i cartelli
V = np.array([[0, 0], [1, 0], [0.5, 0.5]])                 # le merci

punteggi = Q @ K.T                                         # confrontare: nove prodotti scalari
print("la riga di «pesca»:", punteggi[2], "(nel libro 0, 2, 2)")
assert punteggi[2].tolist() == [0, 2, 2]

monete = np.array([softmax(riga) for riga in punteggi]) * 100     # dividere: una softmax per riga
for parola, riga, attesi in zip(parole, monete, [[33, 33, 33], [9, 66, 25], [6, 47, 47]]):
    for verso, v, a in zip(parole, riga, attesi):
        controlla(f"{parola} → {verso} (monete)", v, a, 0.55)

cesti = monete / 100 @ V                                   # raccogliere
controlla("il cesto di «pesca»: mare", cesti[2, 0], 0.70)
controlla("il cesto di «pesca»: frutta", cesti[2, 1], 0.23)

con_radice = softmax(punteggi[2] / math.sqrt(2)) * 100
for verso, v, a in zip(parole, con_radice, [11, 45, 45]):
    controlla(f"con la divisione per √2: pesca → {verso}", v, a, 0.5)

la riga di «pesca»: [0. 2. 2.] (nel libro 0, 2, 2)
✓  ieri → ieri (monete): calcolato 33.33, nel libro 33
✓  ieri → barca (monete): calcolato 33.33, nel libro 33
✓  ieri → pesca (monete): calcolato 33.33, nel libro 33
✓  barca → ieri (monete): calcolato 9.003, nel libro 9
✓  barca → barca (monete): calcolato 66.52, nel libro 66
✓  barca → pesca (monete): calcolato 24.47, nel libro 25
✓  pesca → ieri (monete): calcolato 6.338, nel libro 6
✓  pesca → barca (monete): calcolato 46.83, nel libro 47
✓  pesca → pesca (monete): calcolato 46.83, nel libro 47
✓  il cesto di «pesca»: mare: calcolato 0.7025, nel libro 0.7
✓  il cesto di «pesca»: frutta: calcolato 0.2342, nel libro 0.23
✓  con la divisione per √2: pesca → ieri: calcolato 10.84, nel libro 11
✓  con la divisione per √2: pesca → barca: calcolato 44.58, nel libro 45
✓  con la divisione per √2: pesca → pesca: calcolato 44.58, nel libro 45


## Il conto del quadrato

*Nel racconto: «raddoppi la pagina e il lavoro dell'attenzione quadruplica».*

In [4]:
for parole_, atteso in [(10, 100), (1_000, 1e6), (2_000, 4e6), (10_000, 100e6), (100_000, 10e9), (1_000_000, 1000e9)]:
    conta(f"{parole_:>9,} parole → confronti".replace(",", "."), parole_ ** 2, atteso)

✓         10 parole → confronti: calcolato 100, nel libro 100
✓      1.000 parole → confronti: calcolato 1.000.000, nel libro 1.000.000
✓      2.000 parole → confronti: calcolato 4.000.000, nel libro 4.000.000
✓     10.000 parole → confronti: calcolato 100.000.000, nel libro 100.000.000
✓    100.000 parole → confronti: calcolato 10.000.000.000, nel libro 10.000.000.000
✓  1.000.000 parole → confronti: calcolato 1.000.000.000.000, nel libro 1.000.000.000.000


## Le teste

*Nel racconto: «una lista per la cucina, una per la barca»; gli allievi che guardano ciascuno qualcuno di diverso.*

In [5]:
d, teste = 768, 12
conta("numeri per testa", d // teste, 64)
conta("le dodici uscite in fila", teste * (d // teste), 768)
controlla("pesi dello strato di attenzione, 4 · 768² (milioni)", 4 * d ** 2 / 1e6, 2.36)

✓  numeri per testa: calcolato 64, nel libro 64
✓  le dodici uscite in fila: calcolato 768, nel libro 768
✓  pesi dello strato di attenzione, 4 · 768² (milioni): calcolato 2.359, nel libro 2.36


## La maschera

*Nel racconto: «chi sta dietro vede chi ha davanti, mai chi ha alle spalle».*

In [6]:
n = 3
M = np.triu(np.full((n, n), -np.inf), k=1)
print(M)
monete_m = np.array([softmax(riga) for riga in punteggi + M])
print(np.round(monete_m * 100))
assert np.allclose(np.triu(monete_m, k=1), 0)              # nessuna moneta alle parole future
controlla("la prima parola guarda solo se stessa", monete_m[0, 0], 1)

[[  0. -inf -inf]
 [  0.   0. -inf]
 [  0.   0.   0.]]
[[100.   0.   0.]
 [ 12.  88.   0.]
 [  6.  47.  47.]]
✓  la prima parola guarda solo se stessa: calcolato 1, nel libro 1


## La bussola che gira

*Nel racconto: «ogni parola gira la sua bussola di un angolo che dipende dal suo posto».*

In [7]:
def ruota(v, posizione, theta=0.1):
    a = posizione * theta
    return np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]]) @ v

lista = cartello = np.array([1.0, 0.0])
controlla("posizioni 5 e 3", float(ruota(lista, 5) @ ruota(cartello, 3)), 0.98)
controlla("posizioni 12 e 10", float(ruota(lista, 12) @ ruota(cartello, 10)), 0.98)
controlla("cos 0,2", math.cos(0.2), 0.98)

✓  posizioni 5 e 3: calcolato 0.9801, nel libro 0.98
✓  posizioni 12 e 10: calcolato 0.9801, nel libro 0.98
✓  cos 0,2: calcolato 0.9801, nel libro 0.98


## Il blocco, e quanti pesi ha

*Nel racconto: il kata ripetuto; il marinaio che ritende la cima; la seconda cima tesa diritta.*

In [8]:
attenzione = 4 * d ** 2
mlp = 2 * d * 4 * d
controlla("attenzione, 4d² (milioni)", attenzione / 1e6, 2.36)
controlla("piccola rete, 8d² (milioni)", mlp / 1e6, 4.72)
controlla("un blocco (milioni)", (attenzione + mlp) / 1e6, 7.1, 0.05)
controlla("dodici blocchi (milioni)", 12 * (attenzione + mlp) / 1e6, 85, 0.5)
controlla("tabella degli embedding, 50.257 × 768 (milioni)", 50257 * d / 1e6, 38.6, 0.05)
controlla("posizioni, 1.024 × 768 (milioni)", 1024 * d / 1e6, 0.8, 0.05)
controlla("in tutto: il più piccolo dei GPT-2 (milioni)", (12 * (attenzione + mlp) + 50257 * d + 1024 * d) / 1e6, 124, 0.6)

✓  attenzione, 4d² (milioni): calcolato 2.359, nel libro 2.36
✓  piccola rete, 8d² (milioni): calcolato 4.719, nel libro 4.72
✓  un blocco (milioni): calcolato 7.078, nel libro 7.1
✓  dodici blocchi (milioni): calcolato 84.93, nel libro 85
✓  tabella degli embedding, 50.257 × 768 (milioni): calcolato 38.6, nel libro 38.6
✓  posizioni, 1.024 × 768 (milioni): calcolato 0.7864, nel libro 0.8
✓  in tutto: il più piccolo dei GPT-2 (milioni): calcolato 124.3, nel libro 124


## La ferramenta nuova dal 2017

*Nel racconto: lo stesso scafo da decenni, con la ferramenta e le cime cambiate.*

In [9]:
controlla("GQA con 32 teste di domanda e 8 gruppi: la memoria per cartelli e merci si divide per", 32 / 8, 4)

✓  GQA con 32 teste di domanda e 8 gruppi: la memoria per cartelli e merci si divide per: calcolato 4, nel libro 4


## Kata per casa

**Esercizio 1.** Lista q = (1; 0), cartelli k₁ = (1; 0) e k₂ = (0; 1), merci 10 e 20.

In [10]:
q = np.array([1, 0]); k1 = np.array([1, 0]); k2 = np.array([0, 1])
punteggi = np.array([q @ k1, q @ k2]) / math.sqrt(2)
controlla("primo punteggio diviso per √2", punteggi[0], 0.707, 0.001)
controlla("e^0,707", math.exp(punteggi[0]), 2.028, 0.001)
pesi = softmax(punteggi)
controlla("peso della prima parola", pesi[0], 0.67)
controlla("peso della seconda parola", pesi[1], 0.33)
controlla("uscita", float(pesi @ [10, 20]), 13.3, 0.05)

✓  primo punteggio diviso per √2: calcolato 0.7071, nel libro 0.707
✓  e^0,707: calcolato 2.028, nel libro 2.028
✓  peso della prima parola: calcolato 0.6698, nel libro 0.67
✓  peso della seconda parola: calcolato 0.3302, nel libro 0.33
✓  uscita: calcolato 13.3, nel libro 13.3


**Esercizio 2.** La pagina passa da 1.000 a 4.000 parole.

In [11]:
conta("elementi della tabella con 1.000 parole", 1000 ** 2, 1_000_000)
conta("elementi della tabella con 4.000 parole", 4000 ** 2, 16_000_000)
conta("quante volte più grande", 4000 ** 2 // 1000 ** 2, 16)

✓  elementi della tabella con 1.000 parole: calcolato 1.000.000, nel libro 1.000.000
✓  elementi della tabella con 4.000 parole: calcolato 16.000.000, nel libro 16.000.000
✓  quante volte più grande: calcolato 16, nel libro 16
